<a href="https://colab.research.google.com/github/Afsar110/Chatting-App/blob/afsar/Copy_of_falcon_7b_invoice_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mistral-7B-Instruct-v0.3 QLoRA Fine-Tuning for Invoice Data Extraction

This notebook fine-tunes Mistral-7B-Instruct-v0.3 using QLoRA to extract structured invoice data from raw OCR text.

**Dataset:** falcon_training_dataset.json (676 high-quality examples)

**Format:** SFT with instruction, input, output fields

---

## Setup Instructions

1. Upload `falcon_training_dataset.json` to Colab
2. Set your HuggingFace token in Colab secrets (key: `HF_TOKEN`)
3. Run all cells

## Install Dependencies

⚠️ **Using pinned versions for compatibility**

In [ ]:
# Fix locale encoding issue
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [ ]:
# Install with pinned versions for compatibility
# !pip install -q bitsandbytes==0.41.1
!pip install -U bitsandbytes
!pip install -U transformers
!pip install -U peft==0.7.1
!pip install -U accelerate
!pip install -U einops



In [ ]:
# # Restart runtime after installing (run this cell and then restart)
# import os
# os._exit(00)  # This will restart the runtime

⚠️ **After running the cell above, the runtime will restart. Continue from the next cell.**

In [ ]:
# Re-apply locale fix after restart
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

# Verify versions
import transformers
import peft
print(f"transformers version: {transformers.__version__}")
print(f"peft version: {peft.__version__}")

transformers version: 4.57.5
peft version: 0.7.1


## Import Libraries

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import transformers
import torch
from torch.utils.data import Dataset

## Load Quantized Model

Using the official Falcon-7B-Instruct model from TII UAE

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=False   # 🔥 critical
)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully!")


Loading tokenizer...
Loading model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully!


## Setup LoRA Configuration

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

def print_trainable_parameters(model):
    """Prints the number of trainable parameters in the model."""
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params:,} || all params: {all_param:,} || trainable%: {100 * trainable_params / all_param:.2f}")

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


model = get_peft_model(model, config)
print_trainable_parameters(model)

trainable params: 6,815,744 || all params: 3,765,178,368 || trainable%: 0.18


## Upload and Load Training Data

⚠️ **Make sure to upload `falcon_training_dataset.json` before running this cell!**

Click the folder icon on the left → Upload → Select your JSON file

In [ ]:
import json
import random

# Load the prepared invoice training dataset
json_path = '/content/falcon_training_dataset.json'

print(f"Loading training data from: {json_path}")
with open(json_path, 'r', encoding='utf-8') as f:
    all_data = json.load(f)
print(f"✓ Loaded {len(all_data)} training examples")

# Split into train and validation sets (90/10 split)
random.seed(42)  # For reproducibility
random.shuffle(all_data)

val_size = max(1, int(len(all_data) * 0.1))  # 10% for validation
train_data = all_data[:-val_size]
val_data = all_data[-val_size:]

print(f"✓ Training set: {len(train_data)} examples")
print(f"✓ Validation set: {len(val_data)} examples")

Loading training data from: /content/falcon_training_dataset.json
✓ Loaded 676 training examples
✓ Training set: 609 examples
✓ Validation set: 67 examples


## Prepare Dataset for Training

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

# Format data for instruction fine-tuning
def format_sft_example(example):
    """Format a single example for supervised fine-tuning."""
    return f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""

train_texts = [format_sft_example(ex) for ex in train_data]
val_texts = [format_sft_example(ex) for ex in val_data]

# Tokenize the datasets
print("Tokenizing training data...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=2048, return_tensors='pt')
print("Tokenizing validation data...")
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=2048, return_tensors='pt')

class TextDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = item["input_ids"].clone()
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

# Convert the encodings to PyTorch datasets
train_dataset = TextDataset(train_encodings)
val_dataset = TextDataset(val_encodings)
print(f"✓ Training dataset: {len(train_dataset)} examples")
print(f"✓ Validation dataset: {len(val_dataset)} examples")

Tokenizing training data...
Tokenizing validation data...
✓ Training dataset: 609 examples
✓ Validation dataset: 67 examples


## Test Model Before Fine-Tuning (Optional)

In [ ]:
def generate_invoice_extraction(index):
    """Generate invoice extraction for a given training example."""
    if index >= len(train_data):
        print(f"Index {index} out of range. Dataset has {len(train_data)} examples.")
        return

    example = train_data[index]

    input_text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input'][:1000]}{'...' if len(example['input']) > 1000 else ''}

### Response:"""

    expected_response = example['output']

    print("=" * 60)
    print("INPUT (truncated):")
    print(input_text[:500] + "..." if len(input_text) > 500 else input_text)
    print("=" * 60)

    encoding = tokenizer(input_text, return_tensors="pt").to("cuda:0")
    output = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    print("\nGENERATED RESPONSE:")
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    if '### Response:' in generated:
        generated = generated.split('### Response:')[1].strip()
    print(generated[:1000] if len(generated) > 1000 else generated)

    print("\nEXPECTED RESPONSE (truncated):")
    print(expected_response[:500] if len(expected_response) > 500 else expected_response)
    print("=" * 60)

# Test before training (optional - can skip to save time)
generate_invoice_extraction(0)

INPUT (truncated):
### Instruction:
SYSTEM:
You are an accounting invoice interpreter for Indian GST invoices.

TASK:
Convert invoice content into ONE accounting table row.
Follow GST rules strictly.
Do not invent values.

### Input:
--- PAGE 1 ---
GST Invoice
Registered office address - SCO-205, GROUND FLOOR, SECTOR-12, KARNAL(132001), HARYANA
Tel No: +91-9813112555, Email Address: no-reply@etradeonline.in, Website: N/A
CIN: U72200HR2010PTC041259, PAN: AADCV4254H
Page 1 of 6
Billing Address
Receiver Billing Addre...

GENERATED RESPONSE:
| Invoice Number | Invoice Date | Supplier Name | Supplier GSTIN | Supplier State Code | Supplier Address Line 1 | Supplier Address Line 2 | Supplier Address Line 3 | Supplier Address Line 4 | Supplier Address Line 5 | Supplier Address Line 6 |
|---------------|--------------|---------------|------------------|---------------------|------------------------|------------------------|------------------------|------------------------|----------------------

## Training Configuration

In [ ]:
# Training configuration optimized for 676 examples
batch_size = 2  # Good balance for GPU memory and training stability
num_epochs = 5  # 676 examples * 10 epochs
gradient_accumulation = 4  # Effective batch size = 4 * 4 = 16

print(f"{'='*60}")
print(f"TRAINING CONFIGURATION")
print(f"{'='*60}")
print(f"Batch size: {batch_size}")
print(f"Gradient accumulation: {gradient_accumulation}")
print(f"Effective batch size: {batch_size * gradient_accumulation}")
print(f"Number of epochs: {num_epochs}")
print(f"Total training examples: {len(train_dataset)}")
print(f"Total validation examples: {len(val_dataset)}")
print(f"{'='*60}")

TRAINING CONFIGURATION
Batch size: 2
Gradient accumulation: 4
Effective batch size: 8
Number of epochs: 5
Total training examples: 609
Total validation examples: 67


## Start Training

⏱️ **This will take approximately 2-4 hours on a T4 GPU**

In [ ]:
trainer = transformers.Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=transformers.TrainingArguments(
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation,
        warmup_ratio=0.05,
        learning_rate=1e-4,
        fp16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        output_dir="outputs",
        optim="paged_adamw_8bit",
        lr_scheduler_type='cosine',
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

model.config.use_cache = False
print("\n🚀 Starting training...")
trainer.train()
print("\n✅ Training complete!")


🚀 Starting training...


/tmp/ipython-input-3045687004.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,0.522100,0.479323
100,0.345500,0.369405
150,0.329600,0.323929


/tmp/ipython-input-3045687004.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


## Test Model After Fine-Tuning

In [ ]:
model.config.use_cache = True
model.eval()

if len(train_data) > 0:
    generate_invoice_extraction(0)

## Save Model to HuggingFace Hub

⚠️ **Make sure to set your HF_TOKEN in Colab Secrets before running!**

Go to: 🔑 (key icon) → Add new secret → Name: `HF_TOKEN` → Value: Your token

In [ ]:
# ============================================================
# CONFIGURE YOUR MODEL NAMES HERE
# ============================================================
HF_USERNAME = "afsaransari110"  # Your HuggingFace username
ADAPTERS_REPO = f"{HF_USERNAME}/falcon-7b-invoice-extractor-lora"
MERGED_REPO = f"{HF_USERNAME}/falcon-7b-invoice-extractor"

print(f"\n{'='*60}")
print("MODEL SAVING CONFIGURATION")
print(f"{'='*60}")
print(f"Adapters will be saved to: {ADAPTERS_REPO}")
print(f"Merged model will be saved to: {MERGED_REPO}")
print(f"{'='*60}")

In [ ]:
# Login to HuggingFace
from huggingface_hub import login
import os

# Get token from Colab Secrets
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("✅ Logged in to HuggingFace Hub")
else:
    from huggingface_hub import notebook_login
    print("⚠️ HF_TOKEN not found in secrets. Using interactive login...")
    notebook_login()

In [ ]:
# Save LoRA adapters
print("\n[Step 1/3] Saving LoRA adapters...")
model.save_pretrained("falcon-7b-invoice-extractor-lora")
model.push_to_hub(ADAPTERS_REPO)
print(f"✅ Adapters saved to: {ADAPTERS_REPO}")

In [ ]:
# Reload base model and merge with adapters
print("\n[Step 2/3] Reloading base model for merging...")
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-7b-instruct",
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

merged_model = PeftModel.from_pretrained(base_model, "falcon-7b-invoice-extractor-lora")
merged_model = merged_model.merge_and_unload()
print("✅ Model merged successfully")

In [ ]:
# Push merged model to hub
print("\n[Step 3/3] Pushing merged model to Hub...")
merged_model.push_to_hub(MERGED_REPO)

# Also push tokenizer
tokenizer_final = AutoTokenizer.from_pretrained("tiiuae/falcon-7b-instruct", trust_remote_code=True)
tokenizer_final.push_to_hub(MERGED_REPO)

print(f"\n{'='*60}")
print("🎉 TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"LoRA Adapters: https://huggingface.co/{ADAPTERS_REPO}")
print(f"Merged Model: https://huggingface.co/{MERGED_REPO}")
print(f"{'='*60}")